In [ ]:
# TODO: convert a lot of these ipynb into py for use in the osgpool

In [ ]:
from datetime import datetime
import geopandas as gpd
import os
import sys
import shutil
from pathlib import Path
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import rasterio as rio
from transboundary_opera import run1_download_DISP_S1_Static
from opera_utils.disp import _download, _reformat, mintpy
import h5py
from netCDF4 import Dataset
from pyproj import Transformer
import numpy as np
import rioxarray
import time
from requests.exceptions import ConnectionError

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: Look into the best methods to mosaic the MintPy outputs. Joe suggested that using GPS stations to tie the data to is common

In [ ]:
SEARCH_START = datetime(2016, 1, 1)     
SEARCH_END = datetime(2026, 1, 1)

data_storage = Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/')

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
len(gdf)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

for idx, data in gdf.iterrows():
    bbox = data.geometry.bounds
    
    # Plot the geometry (MultiPolygon or Polygon)
    gpd.GeoSeries([data.geometry]).plot(ax=ax, alpha=0.5, edgecolor='k')
    
    # Plot the bounding box as a rectangle
    from shapely.geometry import box
    bbox_geom = box(*bbox)
    gpd.GeoSeries([bbox_geom]).plot(ax=ax, alpha=0.3, edgecolor='r', facecolor='none', linewidth=2)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geometries with Bounding Boxes')
plt.tight_layout()

In [ ]:
frame_ids = dt.get_unique_frame_ids(gdf, track_per_row=True, max_workers=8)
len(frame_ids)

## Step 1. Download the OPERA DISP NetCDF files
- Starting to reach 4 tb capacity for the aquifers. Going to give up and process them individually and then try and remove the no longer needed data.

In [ ]:
# aquifers left over to process as of 09 MAR 2026
# ['N087', 'N026', 'N078', 'N074', 'N071']

list(set(list(gdf.CODE_2021.values)) - set(os.listdir(data_storage)))

In [ ]:
# retries are needed as it is quite common that the connect be interrupted or broken
MAX_RETRIES = 5
RETRY_DELAY = 10  # seconds, will increase exponentially

for idx, data in gdf.iterrows():
    # extract aquifer boundary as simplified bbox
    bbox = data.geometry.bounds

    # extract frames which overlap with aquifer
    frame_ids = data.frame_ids

    # request metadata from ASF to identify the geometry of the OPERA frames
    geom_frames = dt.get_frame_geometries(
            frame_ids, 
            gdf_bounds=data.geometry.bounds
        )

    
    for frame in frame_ids:
        out_path = data_storage / data.CODE_2021 / str(frame)
        subset_dir = out_path / 'subset-ncs'
        
        # Check if downloads already completed (directory exists AND contains .nc files)
        if subset_dir.exists() and list(subset_dir.glob('*.nc')):
            print(f'Data from OPERA frame {frame} for aquifer {data.CODE_2021} already downloaded locally.')
            continue

        os.makedirs(out_path, exist_ok=True)

        # clip the target aquifer by the current opera frame
        clipped = gpd.clip(
            gpd.GeoSeries([data.geometry]),
            geom_frames[geom_frames['frame_id'] == frame]
        )

        # if the clipped aquifer is empty, then the frame does not actually overlap and can be removed and skipped
        if clipped.empty:
            print(f"No overlap for frame {frame}, skipping...")
            shutil.rmtree(out_path)
            continue
        
        clipped_bbox = clipped.geometry.iloc[0].bounds

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                _download.run_download(
                    frame,
                    start_datetime=SEARCH_START,
                    end_datetime=SEARCH_END,
                    num_workers=8,  # reduced from 8
                    output_dir=out_path / 'subset-ncs',
                    bbox=clipped_bbox
                )
                break  # success
            except (ConnectionError, OSError) as e:
                if attempt < MAX_RETRIES:
                    wait = RETRY_DELAY * (2 ** (attempt - 1))
                    print(f"Attempt {attempt} failed for frame {frame}: {e}")
                    print(f"Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"All {MAX_RETRIES} attempts failed for frame {frame}. Skipping.")
                    # optionally clean up partial downloads:
                    # shutil.rmtree(out_path)

## Step 2. Reformat time series stacks for each frame

In [ ]:
# 03 MAR 26: N035 brokey, don't choose that one for now

aquifer = 'N025'

if aquifer:
    print(aquifer)
    print(os.listdir(data_storage / aquifer))
else:
    print('No aquifer selected')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

data = gdf[gdf.CODE_2021 == aquifer]

# Plot the geometry (MultiPolygon or Polygon)
data.geometry.plot(ax=ax, alpha=0.5, edgecolor='k')

# Plot the bounding box as a rectangle
from shapely.geometry import box
total_bounds = data.geometry.total_bounds  # returns (minx, miny, maxx, maxy) as a single array
bbox_geom = box(*total_bounds)
gpd.GeoSeries([bbox_geom]).plot(ax=ax, alpha=0.3, edgecolor='r', facecolor='none', linewidth=2)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geometries with Bounding Boxes')
plt.tight_layout()

In [ ]:
# TODO: try with parallel processing. It took 10 hours to write 2.5 out of 5 files for aquifer N025. That is insane.
# may need to test without corrections, though that would be less accurate deformation estimates

from concurrent.futures import ProcessPoolExecutor, as_completed

In [ ]:
CHUNKING = (64, 512, 512)        # (Time, Height, Width). (4, 256, 256) uses ~ 17GB of my ~90GB, can increase for faster writing
# (64, 1024, 1024) or larger
PROCESS_CHUNKS = (1024, 1024)

DROPPED_VARS = ['phase_similarity', 'persistent_scatterer_mask', 'timeseries_inversion_residuals', 'connected_component_labels', 'estimated_phase_quality', 'shp_counts']


REFERENCE_METHOD = 'high_coherence'     # 'high_coherence' ,'point', or 'border'

# if reference_method is high_coherence, need to set a coherence threshold
# default = 0.7
COHERENCE_THRESHOLD = 0.7

# if reference_method is border, need to set a number of border pixels used 
# default is 0.3
BORDER_PIXELS = 0.3

if aquifer:
    for frame in os.listdir(data_storage / aquifer):
        out_path = data_storage / aquifer / str(frame)
        # os.makedirs(out_path, exist_ok=True)

        # automatically re-references data to the oldest nc file in this list
        # TODO: let a user define custom reformatting. This could look like a list covering different time periods for example
        nc_files = list(out_path.glob('*/*.nc'))
        geomDir = out_path / 'geom_data'
    
        if Path(str(out_path / f'disp-stack-{out_path.stem}.nc')).exists() == True:
            continue
        else:
            if REFERENCE_METHOD == 'point':
                try:
                    test_file = Dataset(nc_files[0])
    
                    temporal_coherence = test_file.variables['temporal_coherence'][:]
                    max_coherence = np.nanmax(temporal_coherence)
    
                    y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
                    y_coord = test_file.variables['y'][y_idx]
                    x_coord = test_file.variables['x'][x_idx]
    
                    test_file.close()
    
                    _reformat.reformat_stack(
                        input_files = nc_files,
                        output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                        out_chunks = CHUNKING,
                        shard_factors = (1, 4, 4),
                        drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                        apply_solid_earth_corrections = True,
                        apply_ionospheric_corrections = True,
                        reference_method = REFERENCE_METHOD,
                        reference_row=y_idx,                                                
                        reference_col=x_idx,
                        process_chunk_size = PROCESS_CHUNKS,
                        do_round = True                                              
                    )
                except Exception as e:
                    print(f"Frame {frame} failed: {e}")
                    continue
                
            elif REFERENCE_METHOD == 'high_coherence':
                try:
                    _reformat.reformat_stack(
                        input_files = nc_files,
                        output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                        out_chunks = CHUNKING,
                        shard_factors = (1, 4, 4),
                        drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                        apply_solid_earth_corrections = True,
                        apply_ionospheric_corrections = True,
                        reference_method = REFERENCE_METHOD,
                        reference_coherence_threshold = COHERENCE_THRESHOLD,
                        process_chunk_size = PROCESS_CHUNKS,
                        do_round = True                               
                    )
                except Exception as e:
                    print(f"Frame {frame} failed: {e}")
                    continue
                
            elif REFERENCE_METHOD == 'border':
                try:
                    _reformat.reformat_stack(
                        input_files = nc_files,
                        output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                        out_chunks = CHUNKING,
                        shard_factors = (1, 4, 4),
                        drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                        apply_solid_earth_corrections = True,
                        apply_ionospheric_corrections = True,
                        reference_method = REFERENCE_METHOD,
                        reference_border_pixels = BORDER_PIXELS,
                        process_chunk_size = PROCESS_CHUNKS,
                        do_round = True                               
                    )
                except Exception as e:
                    print(f"Frame {frame} failed: {e}")
                    continue


else:
    for aquifer in os.listdir(data_storage):
        for frame in os.listdir(data_storage / aquifer):
            out_path = data_storage / aquifer / str(frame)
    
            nc_files = list(out_path.glob('*/*.nc'))
            geomDir = out_path / 'geom_data'

            if Path(str(out_path / f'disp-stack-{out_path.stem}.nc')).exists() == True:
                continue
            else:
                if REFERENCE_METHOD == 'point':
                    try:
                        test_file = Dataset(nc_files[0])

                        temporal_coherence = test_file.variables['temporal_coherence'][:]
                        max_coherence = np.nanmax(temporal_coherence)

                        y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
                        y_coord = test_file.variables['y'][y_idx]
                        x_coord = test_file.variables['x'][x_idx]

                        test_file.close()

                        _reformat.reformat_stack(
                            input_files = nc_files,
                            output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                            out_chunks = CHUNKING,
                            shard_factors = (1, 4, 4),
                            drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                            apply_solid_earth_corrections = True,
                            apply_ionospheric_corrections = True,
                            reference_method = REFERENCE_METHOD,
                            reference_row=y_idx,                                                
                            reference_col=x_idx,
                            process_chunk_size = PROCESS_CHUNKS,
                            do_round = True                                                 
                        )
                    except Exception as e:
                        print(f"Frame {frame} failed: {e}")
                        continue
                    
                elif REFERENCE_METHOD == 'high_coherence':
                    try:
                        _reformat.reformat_stack(
                            input_files = nc_files,
                            output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                            out_chunks = CHUNKING,
                            shard_factors = (1, 4, 4),
                            drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                            apply_solid_earth_corrections = True,
                            apply_ionospheric_corrections = True,
                            reference_method = REFERENCE_METHOD,
                            reference_coherence_threshold = COHERENCE_THRESHOLD,
                            process_chunk_size = PROCESS_CHUNKS,
                            do_round = True                                
                        )
                    except Exception as e:
                        print(f"Frame {frame} failed: {e}")
                        continue
                    
                elif REFERENCE_METHOD == 'border':
                    try:
                        _reformat.reformat_stack(
                            input_files = nc_files,
                            output_name = str(out_path / f'disp-stack-{out_path.stem}.nc'),
                            out_chunks = CHUNKING,
                            shard_factors = (1, 4, 4),
                            drop_vars = DROPPED_VARS,                                                   # can drop variables if we want, but will keep all for future analysis. Can also keep the short wavelength displacement
                            apply_solid_earth_corrections = True,
                            apply_ionospheric_corrections = True,
                            reference_method = REFERENCE_METHOD,
                            reference_border_pixels = BORDER_PIXELS,
                            process_chunk_size = PROCESS_CHUNKS,
                            do_round = True                                
                        )
                    except Exception as e:
                        print(f"Frame {frame} failed: {e}")
                        continue

## Step 3. Download OPERA Static layers for viewing geometry

***Frame 3067 does not have a static layer???***

In [ ]:
if aquifer:
    for frame in os.listdir(data_storage / aquifer):
        out_path = data_storage / aquifer / str(frame)
        frame_nc = out_path / f'disp-stack-{frame}.nc'
        
        staticDir = out_path / 'orbit_data'
        os.makedirs(staticDir, exist_ok=True)
        dispDir = out_path / 'disp_data'
        os.makedirs(dispDir, exist_ok=True)
        geomDir = out_path / 'geom_data'
        os.makedirs(geomDir, exist_ok=True)
        
        sys.argv = [
            'notebook',  # placeholder for script name
            '--frameID', str(frame),
            # '--version', 0.9,
            '--startDate', SEARCH_START.strftime('%Y%m%d'),
            '--endDate', SEARCH_END.strftime('%Y%m%d'),
            '--dispDir', str(dispDir),
            '--staticDir', str(staticDir),
            '--geomDir', str(geomDir),
            '--staticOnly'                             # default is true, uncomment for false
        ]

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                run1_inps = run1_download_DISP_S1_Static.createParser()
                run1_download_DISP_S1_Static.main(run1_inps)
                
            except (ConnectionError, OSError) as e:
                if attempt < MAX_RETRIES:
                    wait = RETRY_DELAY * (2 ** (attempt - 1))
                    print(f"Attempt {attempt} failed for frame {frame}: {e}")
                    print(f"Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"All {MAX_RETRIES} attempts failed for frame {frame}. Skipping.")
                    # optionally clean up partial downloads:
                    # shutil.rmtree(out_path)
        
        losEast = rio.open(geomDir / 'los_east.tif').read(1)
        losNorth = rio.open(geomDir / 'los_north.tif').read(1)
        up = np.sqrt(1 - losEast**2 - losNorth**2)
        
        with rio.open(geomDir / 'los_east.tif') as src:
            crs = src.crs
            transform = src.transform
        
        with rio.open(geomDir / 'los_enu.tif', 'w',
                 driver='GTiff',
                 height=losEast.shape[0],
                 width=losEast.shape[1],
                 count=3,
                 dtype=losEast.dtype,
                 crs=crs,
                 transform=transform) as dst:
            dst.write(losEast, 1)
            dst.write(losNorth, 2)
            dst.write(up, 3)
        
        # clipping the geometry rasters to the same extent as the clipped OPERA DISP frames
        nc_ds = xr.open_dataset(frame_nc)
        for tif in ['layover_shadow_mask.tif', 'los_enu.tif', 'los_east.tif', 'los_north.tif']:

            tiff_ds = rioxarray.open_rasterio(geomDir / tif)

            clipped = tiff_ds.rio.reproject_match(nc_ds)  
            clipped.rio.to_raster(geomDir / tif)

        os.rmdir(dispDir)   # DISP products are already downloaded 

else:
    for aquifer in os.listdir(data_storage):
        for frame in os.listdir(data_storage / aquifer):
            out_path = data_storage / aquifer / str(frame)
            frame_nc = out_path / f'disp-stack-{frame}.nc'

            staticDir = out_path / 'orbit_data'
            os.makedirs(staticDir, exist_ok=True)
            dispDir = out_path / 'disp_data'
            os.makedirs(dispDir, exist_ok=True)
            geomDir = out_path / 'geom_data'
            os.makedirs(geomDir, exist_ok=True)

            sys.argv = [
                'notebook',  # placeholder for script name
                '--frameID', str(frame),
                # '--version', 0.9,
                '--startDate', SEARCH_START.strftime('%Y%m%d'),
                '--endDate', SEARCH_END.strftime('%Y%m%d'),
                '--dispDir', str(dispDir),
                '--staticDir', str(staticDir),
                '--geomDir', str(geomDir),
                '--staticOnly'                             # default is true, uncomment for false
            ]

            run1_inps = run1_download_DISP_S1_Static.createParser()
            run1_download_DISP_S1_Static.main(run1_inps)

            losEast = rio.open(geomDir / 'los_east.tif').read(1)
            losNorth = rio.open(geomDir / 'los_north.tif').read(1)
            up = np.sqrt(1 - losEast**2 - losNorth**2)

            with rio.open(geomDir / 'los_east.tif') as src:
                crs = src.crs
                transform = src.transform

            with rio.open(geomDir / 'los_enu.tif', 'w',
                     driver='GTiff',
                     height=losEast.shape[0],
                     width=losEast.shape[1],
                     count=3,
                     dtype=losEast.dtype,
                     crs=crs,
                     transform=transform) as dst:
                dst.write(losEast, 1)
                dst.write(losNorth, 2)
                dst.write(up, 3)

            # clipping the geometry rasters to the same extent as the clipped OPERA DISP frames
            nc_ds = xr.open_dataset(frame_nc)
            for tif in ['layover_shadow_mask.tif', 'los_enu.tif', 'los_east.tif', 'los_north.tif']:

                tiff_ds = rioxarray.open_rasterio(geomDir / tif)

                clipped = tiff_ds.rio.reproject_match(nc_ds)  
                clipped.rio.to_raster(geomDir / tif)

            os.rmdir(dispDir)   # DISP products are already downloaded 

## Step 4. prepare OPERA DISP data into a MintPy compatible format

In [ ]:
# variable to set percentage of time pixel is present in each epoch
# if below this threshold, pixel data is not saved to mintpy files
# default is 0.9
RELIABILITY_THRESHOLD = 0.9

if aquifer:
    for frame in os.listdir(data_storage / aquifer):
        out_path = data_storage / aquifer / str(frame)
        nc_files = list(out_path.glob('*/*.nc'))
        geomDir = out_path / 'geom_data'

        try:
            mintpy.disp_nc_to_mintpy(
                str(out_path / f'disp-stack-{out_path.stem}.nc'),
                sample_disp_nc = nc_files[0],
                los_enu_path = geomDir / 'los_enu.tif',
                dem_path = None,
                layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
                outdir = out_path / "mintpy"    ,
                virtual = True,
                reliability_threshold = RELIABILITY_THRESHOLD,
            )
        except Exception as e:
            print(f"Frame {frame} failed: {e}")
            continue

else:
    for aquifer in os.listdir(data_storage):
        for frame in os.listdir(data_storage / aquifer):
            out_path = data_storage / aquifer / str(frame)
            nc_files = list(out_path.glob('*/*.nc'))
            geomDir = out_path / 'geom_data'

            try:
                mintpy.disp_nc_to_mintpy(
                    str(out_path / f'disp-stack-{out_path.stem}.nc'),
                    sample_disp_nc = nc_files[0],
                    los_enu_path = geomDir / 'los_enu.tif',
                    dem_path = None,
                    layover_shadow_mask_path = geomDir / 'layover_shadow_mask.tif',
                    outdir = out_path / "mintpy"    ,
                    virtual = True,
                    reliability_threshold = RELIABILITY_THRESHOLD,
                )
            except Exception as e:
                print(f"Frame {frame} failed: {e}")
                continue

### We need to assign a reference location based on coherence so we can decompose 
- This is the location with the highest average temporal coherence

In [ ]:
if aquifer:
        for frame in os.listdir(data_storage / aquifer):
            base = str(data_storage / aquifer / frame / 'mintpy')

            with h5py.File(f'{base}/geometryGeo.h5', 'r') as geom:
                epsg = dict(geom.attrs)['EPSG']
            transformer = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)

            with xr.open_dataset(f'{base}/avgSpatialCoh.h5', engine='h5netcdf') as coh_ds:
                coh_ds = coh_ds.rename({'phony_dim_0': 'y', 'phony_dim_1': 'x'})
                coherence = coh_ds.avgSpatialCoh

                stacked = coherence.stack(z=("y", "x"))
                max_flat_idx = int(stacked.argmax().values)
                y_idx, x_idx = np.unravel_index(max_flat_idx, coherence.shape)

                y_coord = coh_ds["y"].isel(y=y_idx).values.item()
                x_coord = coh_ds["x"].isel(x=x_idx).values.item()

            ref_lon, ref_lat = transformer.transform(x_coord, y_coord)
            ref_lat = float(ref_lat)
            ref_lon = float(ref_lon)
            ref_y = int(y_idx)
            ref_x = int(x_idx)

            attrs = {
                'REF_LAT': ref_lat,
                'REF_LON': ref_lon,
                'REF_Y':   ref_y,
                'REF_X':   ref_x,
            }

            for fname in ['velocity.h5', 'timeseries.h5', 'geometryGeo.h5']:
                fpath = f'{base}/{fname}'
                with h5py.File(fpath, 'r+') as f:
                    f.attrs.update(attrs)
                print(f"Added to {fpath}")
                print(f"  REF_LAT: {ref_lat:.6f}, REF_LON: {ref_lon:.6f}, REF_Y: {ref_y}, REF_X: {ref_x}")

else:
    for aquifer in os.listdir(data_storage):
        for frame in os.listdir(data_storage / aquifer):
            base = str(data_storage / aquifer / frame / 'mintpy')
        
            with h5py.File(f'{base}/geometryGeo.h5', 'r') as geom:
                epsg = dict(geom.attrs)['EPSG']
            transformer = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)
        
            with xr.open_dataset(f'{base}/avgSpatialCoh.h5', engine='h5netcdf') as coh_ds:
                coh_ds = coh_ds.rename({'phony_dim_0': 'y', 'phony_dim_1': 'x'})
                coherence = coh_ds.avgSpatialCoh
        
                stacked = coherence.stack(z=("y", "x"))
                max_flat_idx = int(stacked.argmax().values)
                y_idx, x_idx = np.unravel_index(max_flat_idx, coherence.shape)
        
                y_coord = coh_ds["y"].isel(y=y_idx).values.item()
                x_coord = coh_ds["x"].isel(x=x_idx).values.item()
        
            ref_lon, ref_lat = transformer.transform(x_coord, y_coord)
            ref_lat = float(ref_lat)
            ref_lon = float(ref_lon)
            ref_y = int(y_idx)
            ref_x = int(x_idx)
        
            attrs = {
                'REF_LAT': ref_lat,
                'REF_LON': ref_lon,
                'REF_Y':   ref_y,
                'REF_X':   ref_x,
            }
        
            for fname in ['velocity.h5', 'timeseries.h5', 'geometryGeo.h5']:
                fpath = f'{base}/{fname}'
                with h5py.File(fpath, 'r+') as f:
                    f.attrs.update(attrs)
                print(f"Added to {fpath}")
                print(f"  REF_LAT: {ref_lat:.6f}, REF_LON: {ref_lon:.6f}, REF_Y: {ref_y}, REF_X: {ref_x}")